# PATSTAT Metadata — one row per application in the universe

The twin of `PatentView/notebook/patent_metadata.ipynb`. Primary key `appln_id`, shared with every other
output in `PATSTAT/output/`.

## Raw data
```
raw/tls201  appln            identity, filing year, priority year, family, granted flag, nb_applicants / nb_inventors
raw/tls211  pat_publn        grant year = year of the first publication flagged publn_first_grant = 'Y'
raw/tls209  appln_ipc        IPC main symbol (ipc_position = 'F')
raw/tls224  appln_cpc        CPC symbols -> cpc_code, cpc_code_list, cpc_subclass_list
raw/tls230  appln_techn_field WIPO technology field (largest weight) -> the five sectors
raw/tls207 + raw/tls206      inventors / applicants (person_id), countries, applicant sector (psn_sector)
raw/tls212                   NPL reference count
output/patstat_reference     ref_count = distinct cited applications (patent references, universe, not replenished)
```

## Universe
`ps.UNIVERSE_WHERE`: patents of invention (`ipr_type = 'PI'`), real applications (`appln_id < 900 000 000`),
filing year 1900-2023. Utility models, designs and PATSTAT's artificial applications are out.

## Output columns
`appln_id, appln_auth, appln_kind, appln_nr, filing_year, filing_date, priority_year, publn_year, grant_year, granted,
docdb_family_id, docdb_family_size, inpadoc_family_id, nb_citing_docdb_fam, nb_applicants, nb_inventors,
ref_count, npl_ref_count, ipc_main, cpc_code, cpc_code_list, cpc_subclass_list, techn_field_nr, techn_field, wipo_sector,
inventor_list, applicant_list, inventor_ctry_list, applicant_ctry_list, applicant_sector`

- `filing_year` is the time anchor of every metric here (PatentView: `grant_year`). `grant_year` is NULL for
  the ~half of applications never granted; `publn_year` = earliest publication.
- `cpc_code` = the CPC symbol in the IPC-main subclass if there is one, else the first symbol; symbols are
  normalised (`'G06K   7/0013'` -> `'G06K7/0013'`); lists are `;`-joined, distinct.
- `techn_field_nr` = the WIPO field with the largest weight (ties -> smallest number); `wipo_sector` as in
  PatentView's `g_wipo_technology` (Schmoch concordance).
- `inventor_list` / `applicant_list` = `person_id`s in sequence order; `*_ctry_list` = distinct ISO2 codes.

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PATSTAT')
import ps_common as ps
OUT = ps.OUT
OUT_FP = ps.out('patstat_metadata.parquet')
ps.preflight('patstat_metadata')

REF = ps.out('patstat_reference.parquet')
con = ps.connect()

## 1. Universe + grant year

In [ ]:
%%time
# 1. Universe + grant year
con.execute(f"""CREATE OR REPLACE TABLE base AS
  SELECT appln_id, appln_auth, trim(appln_kind) AS appln_kind, appln_nr,
         appln_filing_year AS filing_year, appln_filing_date AS filing_date,
         NULLIF(earliest_filing_year, 9999) AS priority_year, NULLIF(earliest_publn_year, 9999) AS publn_year,
         granted = 'Y' AS granted, docdb_family_id, docdb_family_size, inpadoc_family_id, nb_citing_docdb_fam,
         nb_applicants, nb_inventors
  FROM {ps.raw('tls201')} WHERE {ps.UNIVERSE_WHERE}""")
con.execute(f"""CREATE OR REPLACE TABLE gy AS
  SELECT appln_id, min(year(publn_date)) AS grant_year FROM {ps.raw('tls211')}
  WHERE publn_first_grant = 'Y' AND publn_date < DATE '9999-01-01' GROUP BY 1""")
n = con.execute('SELECT count(*), count(*) FILTER (WHERE granted) FROM base').fetchone()
print(f'universe: {n[0]:,} applications, {n[1]:,} granted ({n[1]/n[0]*100:.1f}%)')

## 2. References

In [ ]:
%%time
# 2. References: patent references from the edge list, NPL references from tls212 directly
con.execute(f"""CREATE OR REPLACE TABLE refc AS
  SELECT citing_id AS appln_id, count(DISTINCT cited_id) AS ref_count
  FROM read_parquet('{REF}') WHERE NOT replenished GROUP BY 1""")
con.execute(f"""CREATE OR REPLACE TABLE nplc AS
  SELECT p.appln_id, count(DISTINCT c.cited_npl_publn_id) AS npl_ref_count
  FROM {ps.raw('tls212')} c JOIN {ps.raw('tls211')} p USING (pat_publn_id)
  WHERE c.cited_npl_publn_id <> '0' AND c.citn_replenished = 0 GROUP BY 1""")
print(con.execute('SELECT count(*) AS apps_with_patent_refs, round(avg(ref_count), 2) AS mean_ref_count FROM refc').fetchdf().to_string(index=False))
print(con.execute('SELECT count(*) AS apps_with_npl_refs, round(avg(npl_ref_count), 2) AS mean_npl FROM nplc').fetchdf().to_string(index=False))

## 3. Classification

In [ ]:
%%time
# 3. Classification: IPC main, CPC code + lists, WIPO field / sector
con.execute(f"""CREATE OR REPLACE TABLE ipc AS
  SELECT appln_id, min(regexp_replace(ipc_class_symbol, '\\s+', '', 'g')) AS ipc_main
  FROM {ps.raw('tls209')} WHERE ipc_position = 'F' GROUP BY 1""")
con.execute(f"""CREATE OR REPLACE TABLE cpc AS
  WITH s AS (SELECT DISTINCT appln_id, regexp_replace(cpc_class_symbol, '\\s+', '', 'g') AS sym FROM {ps.raw('tls224')}),
       r AS (SELECT s.appln_id, s.sym,
                    row_number() OVER (PARTITION BY s.appln_id ORDER BY (left(s.sym, 4) <> left(i.ipc_main, 4)), s.sym) AS rn
             FROM s LEFT JOIN ipc i USING (appln_id))
  SELECT appln_id, min(sym) FILTER (WHERE rn = 1) AS cpc_code,
         string_agg(sym, ';' ORDER BY sym) AS cpc_code_list,
         string_agg(DISTINCT left(sym, 4), ';' ORDER BY left(sym, 4)) AS cpc_subclass_list
  FROM r GROUP BY 1""")
con.execute(f"""CREATE OR REPLACE TABLE tf AS
  SELECT appln_id, techn_field_nr, {ps.wipo_sector_sql('techn_field_nr')} AS wipo_sector FROM (
    SELECT appln_id, techn_field_nr, row_number() OVER (PARTITION BY appln_id ORDER BY weight DESC, techn_field_nr) AS rn
    FROM {ps.raw('tls230')}) WHERE rn = 1""")
print(con.execute('SELECT count(*) AS with_ipc_main FROM ipc').fetchdf().to_string(index=False))
print(con.execute('SELECT count(*) AS with_cpc FROM cpc').fetchdf().to_string(index=False))
print(con.execute('SELECT wipo_sector, count(*) AS n FROM tf GROUP BY 1 ORDER BY n DESC').fetchdf().to_string(index=False))

## 4. Persons

In [ ]:
%%time
# 4. Persons: inventors, applicants, their countries, the first applicant's sector
con.execute(f"""CREATE OR REPLACE TABLE pers AS
  SELECT pa.appln_id,
         string_agg(pa.person_id, ';' ORDER BY pa.invt_seq_nr)  FILTER (WHERE pa.invt_seq_nr  > 0) AS inventor_list,
         string_agg(pa.person_id, ';' ORDER BY pa.applt_seq_nr) FILTER (WHERE pa.applt_seq_nr > 0) AS applicant_list,
         string_agg(DISTINCT p.person_ctry_code, ';' ORDER BY p.person_ctry_code) FILTER (WHERE pa.invt_seq_nr  > 0 AND trim(p.person_ctry_code) <> '') AS inventor_ctry_list,
         string_agg(DISTINCT p.person_ctry_code, ';' ORDER BY p.person_ctry_code) FILTER (WHERE pa.applt_seq_nr > 0 AND trim(p.person_ctry_code) <> '') AS applicant_ctry_list,
         min(NULLIF(trim(p.psn_sector), '')) FILTER (WHERE pa.applt_seq_nr = 1) AS applicant_sector
  FROM {ps.raw('tls207')} pa
  JOIN base b USING (appln_id)
  JOIN {ps.raw('tls206')} p USING (person_id)
  GROUP BY 1""")
print(con.execute("""SELECT count(*) AS apps_with_persons, count(inventor_list) AS with_inventors, count(applicant_list) AS with_applicants,
  count(inventor_ctry_list) AS with_inventor_country FROM pers""").fetchdf().to_string(index=False))

## 5. Save + describe

In [ ]:
%%time
# 5. Assemble, write, describe
con.execute(f"""COPY (
  SELECT b.appln_id, b.appln_auth, b.appln_kind, b.appln_nr, b.filing_year, b.filing_date, b.priority_year, b.publn_year,
         g.grant_year, b.granted, b.docdb_family_id, b.docdb_family_size, b.inpadoc_family_id, b.nb_citing_docdb_fam,
         b.nb_applicants, b.nb_inventors,
         coalesce(r.ref_count, 0)::INTEGER AS ref_count, coalesce(n.npl_ref_count, 0)::INTEGER AS npl_ref_count,
         i.ipc_main, c.cpc_code, c.cpc_code_list, c.cpc_subclass_list,
         t.techn_field_nr, t.wipo_sector,
         p.inventor_list, p.applicant_list, p.inventor_ctry_list, p.applicant_ctry_list, p.applicant_sector
  FROM base b
  LEFT JOIN gy g USING (appln_id) LEFT JOIN refc r USING (appln_id) LEFT JOIN nplc n USING (appln_id)
  LEFT JOIN ipc i USING (appln_id) LEFT JOIN cpc c USING (appln_id) LEFT JOIN tf t USING (appln_id)
  LEFT JOIN pers p USING (appln_id)
) TO '{OUT_FP}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
meta = pq.ParquetFile(OUT_FP)
print(f'WROTE {OUT_FP}  ({meta.metadata.num_rows:,} rows, {len(meta.schema_arrow.names)} cols, {os.path.getsize(OUT_FP)/1e9:.2f} GB)')
nn = con.execute(f"SELECT {', '.join(f'round(100.0*count({c})/count(*), 1) AS {c}' for c in meta.schema_arrow.names)} FROM read_parquet('{OUT_FP}')").fetchdf().T
nn.columns = ['non_null_pct']; display(nn)
# techn_field name for the reader; kept out of the parquet (a 35-row lookup, ps.WIPO_FIELD)
display(con.execute(f"""SELECT filing_year, count(*) AS n, round(100.0*avg(granted::INT),1) AS pct_granted, round(avg(ref_count),2) AS mean_refs
  FROM read_parquet('{OUT_FP}') WHERE filing_year BETWEEN 1980 AND 2022 AND filing_year % 5 = 0 GROUP BY 1 ORDER BY 1""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE granted AND ref_count > 5 LIMIT 5").fetchdf())
con.close()